# Notebook 4: Training on 3070 Ti

**What:** Run Demucs training with memory-efficient settings for 8GB VRAM.

**Why:** Default config (batch=64, segment=11s) requires ~16GB+ VRAM. We reduce batch, segment, and model size.

**How:** Use Dora+Hydra overrides; or run a minimal training loop in-notebook.

## 1. 3070 Ti Memory Budget

| Component | Estimate |
|-----------|----------|
| Model (HTDemucs 32ch) | ~50M params ≈ 200MB |
| Optimizer states (Adam) | ~2× model |
| Activations (batch×segment) | Dominant |
| **Total** | Keep < 7GB for safety |

## 2. Recommended Overrides (Dora/Hydra)

From repo root, run:

```powershell
cd D:\demucs
conda activate demucs
dora run -d model=htdemucs batch_size=8 dset.segment=6 htdemucs.channels=32 htdemucs.depth=4 htdemucs.t_layers=3 dset.musdb="C:\\path\\to\\musdbhq"
```

Key overrides:
- `batch_size=8` (or 4 if OOM)
- `dset.segment=6` (shorter segments)
- `htdemucs.channels=32`
- `htdemucs.t_layers=3`

## 3. In-Notebook Minimal Training Loop

Synthetic data, one epoch. No MusDB required. Validates that forward + backward works on 3070 Ti.

In [ ]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
import torch.nn.functional as F
from demucs.htdemucs import HTDemucs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sr = 44100
segment_sec = 4
batch = 4
sources = ['drums', 'bass', 'other', 'vocals']

In [ ]:
model = HTDemucs(
    sources=sources,
    audio_channels=2,
    channels=32,
    depth=4,
    t_layers=3,
    t_heads=4,
    segment=segment_sec,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
# Synthetic "mix" and "sources" (no real separation, just overfitting)
def synthetic_batch(batch, segment_samples):
    # Random sources
    src = torch.randn(batch, len(sources), 2, segment_samples, device=device) * 0.3
    mix = src.sum(dim=1)  # mix = sum of sources
    return mix, src

segment_samples = int(sr * segment_sec)
n_steps = 20  # quick sanity check

for step in range(n_steps):
    mix, targets = synthetic_batch(batch, segment_samples)
    pred = model(mix)
    loss = F.l1_loss(pred, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f"Step {step}, loss={loss.item():.4f}")

print("Training loop OK on 3070 Ti!")

In [ ]:
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 4. Full Training with MusDB HQ

1. Download [MusDB HQ](https://zenodo.org/record/3338373)
2. Update `conf/config.yaml` → `dset.musdb: C:\path\to\musdbhq`
3. Run:

   ```powershell
   dora run -d model=htdemucs batch_size=6 dset.segment=6 htdemucs.channels=32 dset.musdb="C:\\data\\musdbhq"
   ```

4. If OOM: reduce `batch_size` to 4 or `dset.segment` to 5.

**Next:** Notebook 5 — Minimal custom model (2-source toy task).